In [22]:
# 0 Setup: imports, constants, label-free claims
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest

from expense_agent.data import load_claims
from expense_agent.paths import GENERATED_DIR


N_ESTIMATORS = 200 # more trees: steadier scores, slower fit
SEED = 42
DATE_COLUMNS =["expense_date", "submitted_date"]
MIN_GROUP_SIZE = 3 # fewer claims: no reliable personal baseline
REVIEW_SHARE = 0.10 # share of claims reviewer can handle
claims_df = load_claims(GENERATED_DIR)
claims_df[DATE_COLUMNS] = claims_df[DATE_COLUMNS].apply(pd.to_datetime)


assert len(claims_df) == 330
assert claims_df["claim_id"].is_unique
assert "violation_type" not in claims_df.columns, "labels leaked"
print(claims_df[DATE_COLUMNS + ["amount", "category"]].dtypes)

expense_date      datetime64[us]
submitted_date    datetime64[us]
amount                   float64
category                     str
dtype: object


In [15]:
# 1a Date features: submission lag and weekend spending
features= pd.DataFrame(index=claims_df.index)

features["days_to_submit"] = (
    claims_df["submitted_date"] - claims_df["expense_date"]
).dt.days
features["is_weekend"] = (
    claims_df["expense_date"].dt.day_of_week >= 5
).astype(int)

assert features["days_to_submit"].ge(0).all(), "submitted before expense"
assert set(features["is_weekend"].unique()) <= {0, 1}
print(features.describe().round(2))

       days_to_submit  is_weekend
count          330.00      330.00
mean            18.48        0.14
std             16.31        0.35
min              2.00        0.00
25%             11.00        0.00
50%             16.00        0.00
75%             20.00        0.00
max            120.00        1.00


In [16]:
# 1b Amount features: how unusual is this amount for its category?
amount = claims_df["amount"]
by_category = amount.groupby(claims_df["category"])
cat_std = by_category.transform("std").replace(0, np.nan)

features["log_amount"] = np.log1p(amount)
features["amount_vs_cat_median"] = amount / by_category.transform("median")
features["amount_z_in_cat"] = (
    (amount - by_category.transform("mean")) / cat_std
).fillna(0.0)

assert np.isfinite(features.to_numpy()).all(), "NaN or inf in features"
median_ratio = features.groupby(claims_df["category"])[
    "amount_vs_cat_median"
].median()
assert np.allclose(median_ratio, 1.0), median_ratio
print(features.describe().round(2))

       days_to_submit  is_weekend  log_amount  amount_vs_cat_median  \
count          330.00      330.00      330.00                330.00   
mean            18.48        0.14        5.24                  1.22   
std             16.31        0.35        1.14                  1.18   
min              2.00        0.00        2.57                  0.05   
25%             11.00        0.00        4.27                  0.69   
50%             16.00        0.00        5.40                  1.00   
75%             20.00        0.00        6.00                  1.38   
max            120.00        1.00        8.41                 10.64   

       amount_z_in_cat  
count           330.00  
mean             -0.00  
std               0.99  
min              -2.05  
25%              -0.55  
50%              -0.23  
75%               0.39  
max               4.32  


In [17]:
# 1c Employee features: is this claim unusual for this person?
employee = claims_df["employee_id"]
expense_month = claims_df["expense_date"].dt.to_period("M")

features["claims_in_month"] = (
    claims_df["claim_id"].groupby([employee, expense_month]).transform("count")
)

by_emp_cat = amount.groupby([employee, claims_df["category"]])
emp_median = by_emp_cat.transform("median")
emp_count = by_emp_cat.transform("count")
features["amount_vs_emp_median"] = (amount / emp_median).where(
    emp_count >= MIN_GROUP_SIZE, 1.0
)

assert np.isfinite(features.to_numpy()).all(), "NaN or inf in features"
assert features["claims_in_month"].ge(1).all()
fallback_share = (emp_count < MIN_GROUP_SIZE).mean()
print(f"personal baseline missing for {fallback_share:.0%} of claims")
print(features[["claims_in_month", "amount_vs_emp_median"]].describe().round(2))



personal baseline missing for 12% of claims
       claims_in_month  amount_vs_emp_median
count           330.00                330.00
mean              8.45                  1.18
std               3.85                  1.05
min               1.00                  0.07
25%               6.00                  0.72
50%               8.00                  1.00
75%              10.00                  1.25
max              17.00                 10.60


In [18]:
# 1d Feature matrix: lock the column list, check redundancy
FEATURE_COLUMNS = [
    "days_to_submit",
    "is_weekend",
    "log_amount",
    "amount_vs_cat_median",
    "amount_z_in_cat",
    "claims_in_month",
    "amount_vs_emp_median",
]
X = features[FEATURE_COLUMNS]


assert X.shape == (330, len(FEATURE_COLUMNS))
assert np.isfinite(X.to_numpy()).all()
print(X.corr().round(2))

                      days_to_submit  is_weekend  log_amount  \
days_to_submit                  1.00        0.19       -0.08   
is_weekend                      0.19        1.00       -0.07   
log_amount                     -0.08       -0.07        1.00   
amount_vs_cat_median           -0.04       -0.02        0.49   
amount_z_in_cat                -0.01        0.05        0.51   
claims_in_month                -0.25       -0.24        0.05   
amount_vs_emp_median           -0.03       -0.01        0.44   

                      amount_vs_cat_median  amount_z_in_cat  claims_in_month  \
days_to_submit                       -0.04            -0.01            -0.25   
is_weekend                           -0.02             0.05            -0.24   
log_amount                            0.49             0.51             0.05   
amount_vs_cat_median                  1.00             0.83             0.05   
amount_z_in_cat                       0.83             1.00             0.03   
claims_

In [20]:
# 2 IsolationForest: learn "normal" from the features alone
iso_forest = IsolationForest(
    n_estimators=N_ESTIMATORS,
    contamination="auto",
    random_state=SEED,
)
iso_forest.fit(X)

# score_samples: higher = more normal; negate so higher = more anomalous
anomaly_score = pd.Series(
    -iso_forest.score_samples(X), index=X.index, name="anomaly_score"
)

# Same seed, same data -> same scores (reproducibility check)
rerun = IsolationForest(n_estimators=N_ESTIMATORS, random_state=SEED).fit(X)
assert np.allclose(-rerun.score_samples(X), anomaly_score)
assert iso_forest.n_features_in_ == len(FEATURE_COLUMNS)
print(anomaly_score.describe().round(3))

count    330.000
mean       0.430
std        0.071
min        0.348
25%        0.379
50%        0.405
75%        0.462
max        0.737
Name: anomaly_score, dtype: float64


In [23]:
# 3 Threshold by review capacity; inspect the top 10 (no labels)
threshold = anomaly_score.quantile(1 - REVIEW_SHARE)
is_anomaly = anomaly_score >= threshold

scored_claims = (
    claims_df[["claim_id", "employee_id", "category", "amount"]]
    .join(features[FEATURE_COLUMNS])
    .assign(anomaly_score=anomaly_score, is_anomaly=is_anomaly)
)

expected_flags = round(len(scored_claims) * REVIEW_SHARE)
assert abs(is_anomaly.sum() - expected_flags) <= 1, is_anomaly.sum()
print(f"threshold {threshold:.3f}: {is_anomaly.sum()} claims flagged")

show = [
    "claim_id",
    "category",
    "amount",
    "anomaly_score",
    "amount_vs_cat_median",
    "amount_vs_emp_median",
    "days_to_submit",
    "is_weekend",
    "claims_in_month",
]
top10 = scored_claims.nlargest(10, "anomaly_score")
print(top10[show].round(2).to_string(index=False))

threshold 0.537: 33 claims flagged
    claim_id         category  amount  anomaly_score  amount_vs_cat_median  amount_vs_emp_median  days_to_submit  is_weekend  claims_in_month
R-E009-03-L3          Airfare 4491.99           0.74                 10.64                 10.60              13           0                9
R-E001-05-L2      Home Office 1196.21           0.72                  5.43                  5.40              36           1                5
R-E006-08-L1          Airfare 4028.90           0.69                  9.54                  7.80              15           0               10
R-E006-04-L2          Airfare 3201.06           0.67                  7.58                  6.19              22           0               14
R-E003-07-L4          Airfare 3281.92           0.66                  7.77                  6.79              22           0                8
R-E004-05-L4            Meals  179.41           0.62                  4.07                  5.16              22 